# 实验三：谁看起来更好？
## PSNR、SSIM 与感知质量 · From Pixel Error to Perceptual Quality

**课程**：未来媒体互联网（Future Media & Internet） &nbsp;|&nbsp; **预计时长**：~10–12 分钟  
**运行环境**：Kaggle Notebook（CPU） &nbsp;|&nbsp; **无需 GPU** &nbsp;|&nbsp; **Internet Off**

---

### 一句话问题

> **两张图的 PSNR 几乎一样，人眼看到的质量也一定一样吗？**

本实验不先讲公式。我们先看图、先判断，再让指标“揭晓答案”。


## 学习目标

完成本实验后，你应该能够：

1. 解释 PSNR 为什么本质上衡量的是 **像素误差**；
2. 解释 SSIM 为什么会进一步关注 **亮度、对比度和局部结构**；
3. 从实验中理解：**Same PSNR ≠ Same Visual Quality**；
4. 读懂 SSIM Map，判断质量损失主要发生在哪里；
5. 理解单帧图像质量指标如何扩展为逐帧视频质量曲线；
6. 知道 PSNR、SSIM、VMAF、LPIPS 分别代表怎样的指标演进思路。


## 运行环境

| 项目 | 设置 |
|---|---|
| 平台 | Kaggle Notebook |
| 计算资源 | CPU |
| GPU | 不需要 |
| Internet | Off |
| 外部数据集 | 不需要 |
| 图像来源 | `skimage.data.camera()` 内置经典测试图 |
| 依赖 | NumPy、Pandas、Matplotlib、scikit-image |

直接 **Run All** 即可运行。


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from skimage import data, img_as_float
from skimage.filters import gaussian
from skimage.metrics import peak_signal_noise_ratio, structural_similarity
from skimage.transform import resize

rng = np.random.default_rng(42)

reference = img_as_float(data.camera())
TARGET_PSNR = 26.0

print("Environment ready")
print(f"Reference image shape: {reference.shape}")
print(f"Target matched PSNR: {TARGET_PSNR:.1f} dB")


# 第一幕：Blind Visual Challenge

下面有一张 Reference 和四种失真后的图像 A / B / C / D。

**先不要看指标。**

请先凭肉眼判断：

> **A、B、C、D 哪张看起来最好？哪张最差？**

可以先在心里排一个顺序：

**Best → ____ → ____ → ____ → Worst**


In [2]:
# Four different distortion mechanisms will be automatically calibrated
# to almost the SAME PSNR.

noise_pattern = rng.normal(0, 1, reference.shape)

def noise_transform(strength):
    return np.clip(reference + strength * noise_pattern, 0, 1)

def blur_transform(strength):
    return gaussian(reference, sigma=strength, preserve_range=True)

small = resize(
    reference, (128, 128),
    order=1, anti_aliasing=True, preserve_range=True
)
blocky_base = resize(
    small, reference.shape,
    order=0, anti_aliasing=False, preserve_range=True
)

def blocking_transform(strength):
    return np.clip(
        (1 - strength) * reference + strength * blocky_base,
        0, 1
    )

def brightness_transform(strength):
    return np.clip(reference + strength, 0, 1)

def match_target_psnr(transform, low, high, target=TARGET_PSNR, iterations=40):
    # Binary-search a monotonic distortion strength to match target PSNR.
    for _ in range(iterations):
        mid = (low + high) / 2
        candidate = transform(mid)
        score = peak_signal_noise_ratio(reference, candidate, data_range=1.0)
        if score > target:
            low = mid
        else:
            high = mid

    strength = (low + high) / 2
    image = transform(strength)
    return strength, image

specs = [
    ("A", "Noise", noise_transform, 0.0, 0.12),
    ("B", "Blur", blur_transform, 0.01, 5.0),
    ("C", "Blocking", blocking_transform, 0.0, 1.0),
    ("D", "Brightness Shift", brightness_transform, 0.0, 0.20),
]

challenge = {}
for letter, name, transform, low, high in specs:
    strength, image = match_target_psnr(transform, low, high)
    challenge[letter] = {
        "name": name,
        "strength": strength,
        "image": image,
    }

fig, axes = plt.subplots(1, 5, figsize=(16, 4))
axes[0].imshow(reference, cmap="gray")
axes[0].set_title("Reference")
axes[0].axis("off")

for ax, letter in zip(axes[1:], ["A", "B", "C", "D"]):
    ax.imshow(challenge[letter]["image"], cmap="gray")
    ax.set_title(letter)
    ax.axis("off")

plt.suptitle("Blind Visual Challenge — rank A / B / C / D before revealing metrics")
plt.tight_layout()
plt.show()


# 第二幕：Reveal PSNR

现在揭晓第一层答案。

四种失真不是随便选的：程序刚才自动调节了各自的失真强度，使它们的 **PSNR 都接近 26 dB**。

也就是说：

> **像素级平均误差几乎一样。**

问题来了：

> **它们看起来真的一样差吗？**


In [3]:
rows = []

for letter in ["A", "B", "C", "D"]:
    item = challenge[letter]
    psnr = peak_signal_noise_ratio(
        reference, item["image"], data_range=1.0
    )
    rows.append({
        "Image": letter,
        "Distortion": item["name"],
        "PSNR (dB)": psnr,
    })

psnr_table = pd.DataFrame(rows)

display(
    psnr_table.style.format({
        "PSNR (dB)": "{:.3f}",
    })
)

spread = psnr_table["PSNR (dB)"].max() - psnr_table["PSNR (dB)"].min()
print(f"PSNR spread across A–D: only {spread:.6f} dB")


## PSNR：它到底测了什么？

PSNR 建立在均方误差（MSE）上：

\[
MSE=\frac{1}{N}\sum_i(x_i-y_i)^2
\]

\[
PSNR=10\log_{10}\frac{MAX^2}{MSE}
\]

因此 PSNR 主要回答的是：

> **“失真图像的像素值，与参考图像平均相差多少？”**

它并不知道这些误差发生在：

- 平坦背景；
- 人脸；
- 边缘；
- 纹理；
- 整体亮度；
- 局部结构。

所以**相同 PSNR 并不保证相同视觉感受**。

这正是下一步引入 SSIM 的动机。


# 第三幕：Reveal SSIM

现在计算 SSIM。

SSIM 不再只看逐像素误差，而是比较局部窗口中的：

- **Luminance（亮度）**
- **Contrast（对比度）**
- **Structure（结构）**

SSIM 越接近 1，说明与参考图像的局部结构越相似。

> 注意：SSIM 的最大值为 1；本实验重点关注“越接近 1 越相似”，而不是把它简单理解成固定的 0–1 百分制。


In [4]:
rows = []
ssim_maps = {}

for letter in ["A", "B", "C", "D"]:
    item = challenge[letter]

    psnr = peak_signal_noise_ratio(
        reference, item["image"], data_range=1.0
    )

    ssim, ssim_map = structural_similarity(
        reference,
        item["image"],
        data_range=1.0,
        full=True
    )

    ssim_maps[letter] = ssim_map

    rows.append({
        "Image": letter,
        "Distortion": item["name"],
        "PSNR (dB)": psnr,
        "SSIM": ssim,
    })

results_df = pd.DataFrame(rows).sort_values(
    "SSIM", ascending=False
).reset_index(drop=True)

display(
    results_df.style.format({
        "PSNR (dB)": "{:.3f}",
        "SSIM": "{:.3f}",
    })
)

best = results_df.iloc[0]
worst = results_df.iloc[-1]

print(
    f"Same PSNR, but SSIM ranges from "
    f"{worst['SSIM']:.3f} ({worst['Distortion']}) "
    f"to {best['SSIM']:.3f} ({best['Distortion']})."
)
print("Key result: Same pixel error ≠ same structural similarity.")


# 第四幕：SSIM Map —— 问题发生在哪里？

一个 SSIM 数字告诉我们“整体有多相似”，但 SSIM Map 还能回答：

> **图像的哪些位置结构被破坏得最明显？**

下面把四种失真的局部 SSIM Map 显示出来。

- 越亮：局部结构越接近 Reference；
- 越暗：局部结构差异越明显。

观察人脸、头发、相机边缘以及平坦背景区域的差异。


In [5]:
fig, axes = plt.subplots(1, 4, figsize=(15, 4))

for ax, letter in zip(axes, ["A", "B", "C", "D"]):
    ax.imshow(ssim_maps[letter], vmin=0, vmax=1)
    score = structural_similarity(
        reference,
        challenge[letter]["image"],
        data_range=1.0
    )
    ax.set_title(
        f"{letter}: {challenge[letter]['name']}\nSSIM={score:.3f}"
    )
    ax.axis("off")

plt.suptitle("Local SSIM Maps")
plt.tight_layout()
plt.show()


# 第五幕：从单帧到“视频质量曲线”

前面的实验本质上是 **Full-reference Image Quality Assessment**。

真实 Video Quality Assessment 还需要考虑时间维度。这里不用外部视频或 FFmpeg，而是做一个最小演示：

- 30 个“视频帧”；
- 正常阶段质量较高；
- 中间 5 帧模拟短时网络/编码质量下降；
- 随后恢复。

我们逐帧计算 PSNR 和 SSIM。

这可以把 Demo2 与 Demo3 连起来：

**带宽/码率变化 → 视觉质量变化 → 逐帧质量指标**


In [6]:
n_frames = 30

# A short transient quality drop.
blur_strength = np.full(n_frames, 0.45)
blur_strength[10:15] = np.array([1.2, 1.8, 2.5, 1.8, 1.2])

frame_psnr = []
frame_ssim = []
frames = []

for strength in blur_strength:
    frame = gaussian(reference, sigma=float(strength), preserve_range=True)
    frames.append(frame)

    frame_psnr.append(
        peak_signal_noise_ratio(reference, frame, data_range=1.0)
    )
    frame_ssim.append(
        structural_similarity(reference, frame, data_range=1.0)
    )

frame_ids = np.arange(n_frames)

plt.figure(figsize=(11, 4))
plt.plot(frame_ids, frame_psnr, marker="o", label="PSNR (dB)")
plt.xlabel("Frame")
plt.ylabel("PSNR (dB)")
plt.title("Frame-wise PSNR — transient quality drop and recovery")
plt.grid(alpha=0.2)
plt.legend()
plt.show()

plt.figure(figsize=(11, 4))
plt.plot(frame_ids, frame_ssim, marker="o", label="SSIM")
plt.xlabel("Frame")
plt.ylabel("SSIM")
plt.ylim(0, 1.02)
plt.title("Frame-wise SSIM — transient quality drop and recovery")
plt.grid(alpha=0.2)
plt.legend()
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, idx, title in zip(
    axes,
    [5, 12, 20],
    ["Before drop", "Quality drop", "Recovered"]
):
    ax.imshow(frames[idx], cmap="gray")
    ax.set_title(
        f"{title}\nFrame {idx}\n"
        f"PSNR={frame_psnr[idx]:.1f} dB | SSIM={frame_ssim[idx]:.3f}"
    )
    ax.axis("off")

plt.tight_layout()
plt.show()


# 从 PSNR 到感知质量：指标为什么不断演进？

可以把质量指标理解成一条逐步增加“感知信息”的路线：

**PSNR**  
→ 主要关注像素误差

**SSIM / MS-SSIM**  
→ 加入局部结构、对比度与多尺度信息

**VMAF**  
→ 融合多种与感知相关的特征，并通过学习模型组合成视频质量分数

**LPIPS**  
→ 使用深度网络特征空间衡量感知相似度

本 Demo 只实际运行 PSNR 与 SSIM，因为它们足以建立最核心的概念：

> **评价视觉质量，不能只问“像素差多少”，还要问“视觉结构改变了什么”。**

延伸阅读：

- `skimage.metrics.structural_similarity`
- Netflix VMAF: https://github.com/Netflix/vmaf
- LPIPS / Perceptual Similarity: https://richzhang.github.io/PerceptualSimilarity/


## 实验局限性与思考

### 本实验有意做了哪些简化？

1. 主实验只比较单帧 Full-reference 图像；
2. Mini Video 仍然是由静态参考图生成的 30 帧教学序列；
3. 失真类型只有 Noise / Blur / Blocking / Brightness Shift；
4. 没有处理真实编码器、运动补偿、场景切换和复杂时间感知；
5. 没有实际运行 VMAF、LPIPS 或主观 MOS 实验。

### 思考题

1. 为什么 **Brightness Shift** 和 **Noise** 可以拥有相同 PSNR，却产生明显不同的 SSIM？
2. 如果只优化 PSNR，编码器是否一定会得到人眼最喜欢的结果？
3. SSIM 也会失败吗？什么情况下“结构相似”仍不能代表真实观看体验？
4. Demo2 中 ABR 降低码率后，Demo3 的哪些指标可能首先发生变化？
5. 如果让真人给 A/B/C/D 排序，如何把主观排序与 PSNR / SSIM 做相关性分析？
6. 下一步怎样使用真实视频帧与 VMAF，把这个实验扩展成完整 VQA Demo？


---

← [实验二：自适应视频流与 QoE 优化](https://www.kaggle.com/code/guopingtan/fmi-demo2-qoe-optimization)
&nbsp;|&nbsp;
🏠 [课程主页 · Course Home](https://www.kaggle.com/code/guopingtan/fmi-course-kaggle-hands-on-lab-start-here)
&nbsp;|&nbsp;
[实验四：神经压缩 vs JPEG →](https://www.kaggle.com/code/guopingtan/fmi-demo4-neural-compression)

**FMI Course · Kaggle Hands-on Lab** &nbsp;|&nbsp; MV-AI Lab · Hohai University
